## 项目：基于Value Iteration的倒立摆全局最优控制

### 模块导入

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, clear_output, HTML
import time
import sys
import io
import os

# pydrake standard tools
from pydrake.all import (
    BasicVector,
    DynamicProgrammingOptions,
    FittedValueIteration,
    PeriodicBoundaryCondition,
    Simulator,
)
from pydrake.examples import PendulumPlant

# interactive widgets
import ipywidgets as widgets

# meshcat (standalone package, used in interactive cell)
from meshcat import Visualizer
import meshcat.geometry as meshcat_geom
import meshcat.transformations as tf_meshcat

print("All imports successful.")
print(f"Python: {sys.version.split()[0]}, NumPy: {np.__version__}")

### 物理参数设置

| 变量 | 值 | 含义 |
|------|-----|------|
| `u_max` | 3.0 | 最大控制力矩 [N·m] |
| `N_theta` / `N_thetadot` | 51 / 51 | 状态网格分辨率 |
| `theta_bins` | linspace(0, 2π, 51) | θ 网格点 |
| `thetadot_bins` | linspace(-10, 10, 51) | θ̇ 网格点 |
| `state_grid` | [set(theta_bins), set(thetadot_bins)] | FittedValueIteration 输入格式 |
| `N_u` | 9 | 控制离散级别 |
| `input_grid` | [set(linspace(-3,3,9))] | 控制网格 |
| `time_step` | 0.01 s | Euler 离散化步长 |
| `discount_factor` | 0.999 | 折损因子 |
| `convergence_tol` | 0.001 | 收敛容差 |
| `Theta_plot` / `Thetadot_plot` | meshgrid | 用于 3D 可视化 |

### 状态空间、输入空间网格化
### 设置4个典型初始状态空间待仿真

In [ ]:
# =============================================================================
# 物理参数 (Drake自带PendulumPlant默认参数 m=1, l=0.5, b=0.1, g=9.81)
# =============================================================================
u_max = 3.0        # 最大力矩 [N*m]

# 状态空间离散化
N_theta = 51        # theta grid points
N_thetadot = 51     # thetadot grid points
theta_bins = np.linspace(0.0, 2.0 * np.pi, N_theta)
thetadot_max = 10.0
thetadot_bins = np.linspace(-thetadot_max, thetadot_max, N_thetadot)

# FittedValueIteration函数使用的的状态网格 (接收集合的列表)
state_grid = [set(theta_bins), set(thetadot_bins)]

# 输出力矩空间离散化
N_u = 9
input_grid = [set(np.linspace(-u_max, u_max, N_u))]

# 时间步长
time_step = 0.01  # [s]

# Value Iteration的衰减因子与收敛容差
discount_factor = 0.999
convergence_tol = 0.001

# 将状态空间网格组合为矩阵用于可视化
Theta_plot, Thetadot_plot = np.meshgrid(theta_bins, thetadot_bins, indexing='ij')

# 仿真参数（预设4组典型初始条件）
sim_duration = 8.0
sim_initial_states = [
    (0.1 * np.pi, 0.0, r"Near hanging ($0.1\pi$)"),
    (0.5 * np.pi, 0.0, r"Horizontal ($0.5\pi$)"),
    (0.9 * np.pi, 0.0, r"Near upright ($0.9\pi$)"),
    (1.5 * np.pi, 0.0, r"$1.5\pi$"),
]

print(f"状态网格: {N_theta} x {N_thetadot} = {N_theta * N_thetadot} ")
print(f"角度分辨率: {2*np.pi/N_theta:.4f} rad = {360/N_theta:.1f} deg")
print(f"角速度分辨率: {2*thetadot_max/(N_thetadot-1):.4f} rad/s")
print(f"输出力矩分辨率: {2*u_max/(N_u-1):.4f} N*m")
print(f"衰减因子γ: {discount_factor}")

### 生成单摆系统实例
$$ml^2\ddot{\theta}+mgl\mathrm{sin}\theta=u-b\dot{\theta}$$
$$
\mathbf{\dot{x}}=f(\mathbf{x},u)
=
\begin{bmatrix}
x_2 \\
 \frac{1}{ml^2}(u-bx_2-mgl\mathrm{sin}x_1)
 \end{bmatrix}
 $$
### 代价函数构建
$$
g(\mathbf{x},u)
=
\mathbf{\~{x}}^{T}\mathbf{Q}\mathbf{\~{x}}
+
\mathbf{u}^{T}\mathbf{R}\mathbf{u}
=Q_{t}(\theta-\pi)^2+Q_{dt}\dot{\theta}^2+Ru^2
$$
选定基线参数 $Q_t=10.0, Q_{dt}=1.0, R=0.1$
### 值迭代算法
$$
\hat{J}^{*}(\mathbf{x}) = \underset{\mathrm{min}}{u}\left[\ell(\mathbf{x},u)+\gamma\hat{J}^{*}(f(\mathbf{x},u))\right]
$$
### 记录代价函数迭代过程与最优策略用于可视化

In [ ]:
# ======================
# 成本函数和估计值迭代
# ======================

# 生成单摆系统实例
plant_vi = PendulumPlant()

# 生成代价函数
def make_quadratic_cost(plant, Q_theta=10.0, Q_thetadot=1.0, R=0.1):
    """Return a Drake-compatible quadratic cost function.

    Args:
        plant: Drake system (used to evaluate input)
        Q_theta: weight on angle error (theta - pi)^2
        Q_thetadot: weight on angular velocity
        R: weight on control effort u^2

    Returns:
        cost_fn(context) -> float
    """
    def cost_fn(context):
        x = context.get_continuous_state_vector().CopyToVector()
        x[0] = x[0] - np.pi  # angle error from upright
        u = plant.EvalVectorInput(context, 0).CopyToVector()
        return float(Q_theta * x[0]**2 + Q_thetadot * x[1]**2 + R * u[0]**2)
    return cost_fn

"""
def make_min_time_cost(plant, threshold=0.05):
       '''Return a minimum-time cost function (0 within threshold, 1 outside).'''
    def cost_fn(context):
        x = context.get_continuous_state_vector().CopyToVector()
        x[0] = x[0] - np.pi
        if x.dot(x) < threshold:
            return 0.0
        return 1.0
    return cost_fn
"""


# ===========================
# 记录变化过程，每隔5次迭代记录1帧
# ===========================
frames = []  # list of (iteration, cost_to_go_2d)

def capture_callback(iteration, mesh, cost_to_go, policy):
    """Called by FittedValueIteration at each iteration.
    Captures the cost-to-go surface every 5 iterations for the GIF.
    """
    if iteration % 5 != 0:
        return
    J = np.reshape(cost_to_go.copy(), (N_theta, N_thetadot))
    frames.append((iteration, J))


# --- 配置迭代参数 ---
simulator_vi = Simulator(plant_vi)
options = DynamicProgrammingOptions()

# 角度周期映射
options.periodic_boundary_conditions = [
    PeriodicBoundaryCondition(0, 0.0, 2.0 * np.pi),
]

options.discount_factor = discount_factor
options.convergence_tol = convergence_tol
options.visualization_callback = capture_callback

# 生成代价函数 (基线参数R = 0.1)
cost_baseline = make_quadratic_cost(plant_vi, Q_theta=10.0, Q_thetadot=1.0, R=0.1)

print("Running FittedValueIteration with barycentric interpolation...")
print(f"  Grid: {N_theta} x {N_thetadot} = {N_theta * N_thetadot} states")
print(f"  Control: {N_u} levels, discount = {discount_factor}")
t0 = time.time()

policy, cost_to_go = FittedValueIteration(
    simulator_vi, cost_baseline,
    state_grid, input_grid, time_step, options
)

t_elapsed = time.time() - t0
J_star = np.reshape(cost_to_go, (N_theta, N_thetadot))

print(f"\n 在 {len(frames) * 5} 次迭代后收敛 ({t_elapsed:.1f}s)")
print(f"记录 {len(frames)} 帧用于动画生成")
print(f"J* 值域: [{J_star.min():.3f}, {J_star.max():.3f}]")
print(f"接近 (pi, 0) 处 J* 值 : {J_star[N_theta//2, N_thetadot//2]:.3f}")
print(f"策略类型: {type(policy).__name__}")
print(f"  -> 在单纯形网格上使用重心插值")

### 生成反映代价函数迭代变动的动态图片

In [ ]:
# =============================================================================
# GIF动画：价值迭代过程中3D代价函数曲面的演变
# =============================================================================


print("生成收敛动画...")

# Sample frames evenly (max ~50 frames for reasonable GIF size)
n_total = len(frames)
indices = np.linspace(0, n_total - 1, min(50, n_total), dtype=int)
sampled_frames = [frames[i] for i in indices]

# Global color scale (use final J* range for consistent coloring)
vmin = 0
vmax = J_star.max()

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

def make_frame(frame_idx):
    ax.clear()
    iteration, J = sampled_frames[frame_idx]

    surf = ax.plot_surface(
        Theta_plot, Thetadot_plot, J,
        cmap=cm.viridis, linewidth=0, antialiased=True,
        alpha=0.9, vmin=vmin, vmax=vmax
    )
    ax.set_xlabel(r'$\theta$ [rad]', fontsize=10)
    ax.set_ylabel(r'$\dot{\theta}$ [rad/s]', fontsize=10)
    ax.set_zlabel(r'$J(\theta, \dot{\theta})$', fontsize=10)
    ax.set_title(f'Value Iteration Convergence — Iteration {iteration}',
                 fontsize=14, pad=15)

    # Mark target position
    ax.scatter([np.pi], [0], [vmax * 0.02], color='red', s=50, marker='o',
               label='Target (pi, 0)')
    ax.legend(fontsize=9)

    # Consistent viewing angle
    ax.view_init(elev=25, azim=-60)

    return [surf]

# Create animation
ani = animation.FuncAnimation(
    fig, make_frame, frames=len(sampled_frames),
    interval=200, blit=False, repeat=True
)

# Save as GIF
gif_path = 'vi_convergence.gif'
ani.save(gif_path, writer='pillow', fps=5, dpi=100)
plt.close(fig)

print(f"GIF saved: {gif_path}  ({len(sampled_frames)} 帧)")

# Display inline
from IPython.display import Image as IPImage
display(IPImage(filename=gif_path))

# Also show first and last frame side by side for comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                subplot_kw={'projection': '3d'})

for ax, (iteration, J), title in [
    (ax1, frames[0], f'Initial (iter {frames[0][0]})'),
    (ax2, frames[-1], f'Converged (iter {frames[-1][0]})'),
]:
    surf = ax.plot_surface(Theta_plot, Thetadot_plot, J,
                            cmap=cm.viridis, linewidth=0,
                            antialiased=True, alpha=0.9)
    ax.set_xlabel(r'$\theta$')
    ax.set_ylabel(r'$\dot{\theta}$')
    ax.set_title(title)
    ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

### 展示最优代价函数与最优控制策略

In [ ]:
# =============================================================================
# 静态可视化: 迭代 J* 结果&最优控制策略
# =============================================================================

# --- 提取策略值 ---
# BarycentricMeshSystem（重心插值）类策略提供了 get_output_values() 生成 flat array
# 化点为面
policy_flat = policy.get_output_values()
policy_values = np.reshape(policy_flat, (N_theta, N_thetadot))

unique_ctrl = np.unique(policy_values)
bangbang_frac = np.sum(np.abs(policy_values) > 0.95 * u_max) / policy_values.size

print(f"使用的控制策略: {np.sort(unique_ctrl)}")
print(f"使用的控制策略数: {len(unique_ctrl)}")
print(f"Bang-bang成分占比: {bangbang_frac:.1%}")

# --- Fig 1: J* 3D曲面 + 等高线 ---
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(Theta_plot, Thetadot_plot, J_star, cmap=cm.viridis,
                         linewidth=0, antialiased=True, alpha=0.9)
ax1.set_xlabel(r'$\theta$ [rad]')
ax1.set_ylabel(r'$\dot{\theta}$ [rad/s]')
ax1.set_title(r'Optimal Cost-to-Go $J^*(\theta, \dot{\theta})$')
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

ax2 = fig.add_subplot(1, 2, 2)
ct = ax2.contourf(Theta_plot, Thetadot_plot, J_star, levels=30, cmap=cm.viridis)
ax2.axvline(x=np.pi, color='r', linestyle='--', alpha=0.5, label=r'Target ($\pi$)')
ax2.set_xlabel(r'$\theta$ [rad]')
ax2.set_ylabel(r'$\dot{\theta}$ [rad/s]')
ax2.set_title(r'$J^*$ Contours')
ax2.legend()
fig.colorbar(ct, ax=ax2, shrink=0.8)
plt.tight_layout()
plt.savefig('value_function.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fig 2: 策略热力图 ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(Theta_plot, Thetadot_plot, policy_values,
                    cmap=cm.RdBu, shading='auto', vmin=-u_max, vmax=u_max)
ax.axvline(x=np.pi, color='k', linestyle='--', alpha=0.5, label=r'Target ($\pi$)')
ax.set_xlabel(r'$\theta$ [rad]')
ax.set_ylabel(r'$\dot{\theta}$ [rad/s]')
ax.set_title(r'Optimal Policy $\pi^*(\theta, \dot{\theta})$ (R=%.3f,  $\gamma$=%.3f)'
             % (0.1, discount_factor))
ax.legend()
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r'$u$ [N$\cdot$m]')
plt.tight_layout()
plt.savefig('optimal_policy.png', dpi=150, bbox_inches='tight')
plt.show()


### 基线闭环仿真

In [ ]:
# =============================================================================
# 使用重心插值策略的闭环仿真
# =============================================================================

def evaluate_policy(policy_system, theta, thetadot):
    """Evaluate the BarycentricMeshSystem policy at a given state.

    Creates a temporary context to query the policy output.
    Args:
        policy_system: a BarycentricMeshSystem
        theta: wrapped angle [rad]
        thetadot: angular velocity [rad/s]
    Returns:
        u: control torque [N*m]
    """
    ctx = policy_system.CreateDefaultContext()
    policy_system.get_input_port(0).FixValue(ctx, BasicVector([theta, thetadot]))
    return float(policy_system.get_output_port(0).Eval(ctx)[0])


def simulate_closed_loop(policy_system, theta0, thetadot0=0.0,
                         duration=5.0, dt=0.001):
    """Run closed-loop simulation, step by step.

    At each step: evaluate policy -> fix plant input -> integrate for dt.

    Args:
        policy_system: BarycentricMeshSystem from FittedValueIteration
        theta0: initial angle [rad]
        thetadot0: initial angular velocity [rad/s]
        duration: simulation duration [s]
        dt: integration time step [s]

    Returns:
        (t_arr, theta_arr, thetadot_arr, u_arr)
    """
    n_steps = int(duration / dt) + 1
    t_arr = np.zeros(n_steps)
    theta_arr = np.zeros(n_steps)
    thetadot_arr = np.zeros(n_steps)
    u_arr = np.zeros(n_steps)

    x = np.array([theta0, thetadot0])
    theta_arr[0] = theta0
    thetadot_arr[0] = thetadot0

    # 计算初始u
    u = evaluate_policy(policy_system, np.mod(theta0, 2*np.pi),
                        np.clip(thetadot0, -thetadot_max, thetadot_max))
    u_arr[0] = u

    for k in range(n_steps - 1):
        # 创建临时系统，维持dt长的恒定输出并仿真dt时间（模仿离散时间系统行为）
        plant = PendulumPlant()
        context = plant.CreateDefaultContext()
        context.SetContinuousState(x)
        plant.get_input_port(0).FixValue(context, BasicVector([u]))

        sim = Simulator(plant, context)
        try:
            sim.get_mutable_integrator().set_fixed_step_mode(True)
        except AttributeError:
            pass
        sim.AdvanceTo(dt)

        # 读取当前状态
        ctx = sim.get_context()
        x_new = ctx.get_continuous_state_vector().CopyToVector()

        # 循环映射角度
        theta_w = np.mod(x_new[0], 2*np.pi)
        thetadot_c = np.clip(x_new[1], -thetadot_max, thetadot_max)
        u_new = evaluate_policy(policy_system, theta_w, thetadot_c)

        # 记录
        t_arr[k + 1] = (k + 1) * dt
        theta_arr[k + 1] = x_new[0]
        thetadot_arr[k + 1] = x_new[1]
        u_arr[k + 1] = u_new

        x = x_new
        u = u_new

    return t_arr, theta_arr, thetadot_arr, u_arr


# =============================================================================
# 运行4个典型初始条件的倒立摆闭环仿真
# =============================================================================
print("Running closed-loop simulations...\n")

results = {}
for th0, td0, label in sim_initial_states:
    t_arr, th_arr, td_arr, u_arr = simulate_closed_loop(
        policy, th0, td0, duration=5.0
    )
    results[label] = (t_arr, th_arr, td_arr, u_arr)

    err = np.mod(th_arr[-1] - np.pi + np.pi, 2*np.pi) - np.pi
    print(f"  {label}: final theta={th_arr[-1]:.3f}, "
          f"angle_error={abs(err):.4f} rad = {np.degrees(abs(err)):.1f} deg")

# =============================================================================
# 绘制轨迹
# =============================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, (label, (t_arr, th_arr, td_arr, u_arr)) in enumerate(results.items()):
    c = colors[idx]
    axes[0].plot(t_arr, th_arr, color=c, linewidth=1.5, label=label)
    axes[0].axhline(y=np.pi, color='k', linestyle='--', alpha=0.3)

    axes[1].plot(t_arr, td_arr, color=c, linewidth=1.5)

    axes[2].plot(t_arr, u_arr, color=c, linewidth=1.5)
    axes[2].axhline(y=u_max, color='r', linestyle='--', alpha=0.3)
    axes[2].axhline(y=-u_max, color='r', linestyle='--', alpha=0.3)

axes[0].set_ylabel(r'$\theta$ [rad]')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Closed-Loop Simulation with FittedValueIteration Policy '
                  r'($\gamma$=%.3f)' % discount_factor)

axes[1].set_ylabel(r'$\dot{\theta}$ [rad/s]')
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel(r'$u$ [N$\cdot$m]')
axes[2].set_xlabel('Time [s]')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('simulation_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

### 对控制输出成本R进行消融实验

In [ ]:
# =============================================================================
# 消融实验: 改变 R (控制输出成本) 观察bang-bang与平滑控制成分变化
# =============================================================================

R_values = [0.001, 0.01, 0.1, 1.0, 10.0]
ablation_results = {}

for R_val in R_values:
    print(f"\n{'='*60}")
    print(f"FittedValueIteration with R = {R_val}")
    print(f"{'='*60}")

    cost_fn = make_quadratic_cost(plant_vi, Q_theta=10.0, Q_thetadot=1.0, R=R_val)

    opts = DynamicProgrammingOptions()
    opts.periodic_boundary_conditions = [
        PeriodicBoundaryCondition(0, 0.0, 2.0 * np.pi),
    ]
    opts.discount_factor = discount_factor
    opts.convergence_tol = convergence_tol
    # No visualization callback for speed

    sim_vi = Simulator(plant_vi)
    t0 = time.time()
    pol_r, ctg_r = FittedValueIteration(
        sim_vi, cost_fn, state_grid, input_grid, time_step, opts
    )
    elapsed = time.time() - t0

    ctg_2d = np.reshape(ctg_r, (N_theta, N_thetadot))
    pol_flat = pol_r.get_output_values()
    pol_2d = np.reshape(pol_flat, (N_theta, N_thetadot))

    unique_u = np.unique(pol_2d)
    bb_frac = np.sum(np.abs(pol_2d) > 0.95 * u_max) / pol_2d.size

    ablation_results[R_val] = {
        'policy': pol_r,
        'policy_values': pol_2d,
        'cost_to_go': ctg_2d,
        'unique_u': unique_u,
        'bangbang_fraction': bb_frac,
        'elapsed': elapsed,
    }

    print(f"  历时: {elapsed:.1f}s")
    print(f"  使用的输出力矩数: {len(unique_u)}, Bang-bang成分比例: {bb_frac:.1%}")
    print(f"  使用的输出力矩: {np.sort(unique_u)}")

### 可视化不同R值影响

In [ ]:
# =============================================================================
# Fig 4: 不同R值策略分布比较
# =============================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, R_val in enumerate(R_values):
    ax = axes[idx]
    pv = ablation_results[R_val]['policy_values']
    bb = ablation_results[R_val]['bangbang_fraction']
    im = ax.pcolormesh(Theta_plot, Thetadot_plot, pv, cmap=cm.RdBu,
                        shading='auto', vmin=-u_max, vmax=u_max)
    ax.set_xlabel(r'$\theta$ [rad]')
    ax.set_ylabel(r'$\dot{\theta}$ [rad/s]')
    ax.set_title(f'R = {R_val}  (BB = {bb:.0%})')
    ax.axvline(x=np.pi, color='k', linestyle='--', alpha=0.4)

axes[-1].set_visible(False)
fig.colorbar(im, ax=axes[:-1], shrink=0.6, label=r'$u$ [N$\cdot$m]',
             orientation='horizontal', pad=0.08)
fig.suptitle(r'Ablation Study: Policy $\pi^*(\theta,\dot{\theta})$ vs R '
             r'($\gamma$=%.3f)' % discount_factor, fontsize=14)
plt.tight_layout()
plt.savefig('ablation_policies.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================================================
# Figure 5: Metrics vs R
# =============================================================================
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

R_arr = np.array(R_values)
bb_arr = 100 * np.array([ablation_results[r]['bangbang_fraction'] for r in R_values])
nu_arr = np.array([len(ablation_results[r]['unique_u']) for r in R_values])
t_arr = np.array([ablation_results[r]['elapsed'] for r in R_values])

axs[0].semilogx(R_arr, bb_arr, 'bo-', markersize=8)
axs[0].set_xlabel('R'); axs[0].set_ylabel('Bang-bang [%]')
axs[0].set_title('Bang-bang Usage vs R'); axs[0].grid(True, alpha=0.3)

axs[1].semilogx(R_arr, nu_arr, 'rs-', markersize=8)
axs[1].set_xlabel('R'); axs[1].set_ylabel('Unique control values')
axs[1].set_title('Control Diversity vs R'); axs[1].grid(True, alpha=0.3)

axs[2].semilogx(R_arr, t_arr, 'g^-', markersize=8)
axs[2].set_xlabel('R'); axs[2].set_ylabel('Time [s]')
axs[2].set_title('Computation Time vs R'); axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================================================
# Figure 6: Trajectories from same initial state, different R
# =============================================================================
th0_test, td0_test = 0.1 * np.pi, 0.0

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for R_val in R_values:
    pol_r = ablation_results[R_val]['policy']
    t_a, th_a, td_a, u_a = simulate_closed_loop(
        pol_r, th0_test, td0_test, duration=3.0
    )
    axs[0].plot(t_a, np.mod(th_a + np.pi, 2*np.pi) - np.pi,
                linewidth=1.5, alpha=0.8, label=f'R={R_val}')
    axs[1].plot(t_a, td_a, linewidth=1.5, alpha=0.8)
    axs[2].plot(t_a, u_a, linewidth=1.5, alpha=0.8)

axs[0].set_ylabel(r'$\theta - \pi$ [rad]')
axs[0].legend(loc='upper right', fontsize=7, ncol=5)
axs[0].grid(True, alpha=0.3)
axs[0].set_title(fr'Simulation from $\theta_0={th0_test/np.pi:.1f}\pi$ '
                 f'for Different R')

axs[1].set_ylabel(r'$\dot{\theta}$ [rad/s]'); axs[1].grid(True, alpha=0.3)

axs[2].set_ylabel(r'$u$ [N$\cdot$m]'); axs[2].set_xlabel('Time [s]')
axs[2].grid(True, alpha=0.3)
axs[2].axhline(y=u_max, color='k', linestyle='--', alpha=0.3)
axs[2].axhline(y=-u_max, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

### Meshcat交互式3D仿真
使用时同时打开jupyter页面和Meshcat页面（localhost:7000/static/）
操作jupyter页面的组件，观察Meshcat中摆的动作

In [ ]:
# =============================================================================
# 交互式3D仿真
# =============================================================================
import time as _time

# ---- Policy evaluator (same as cell_06, directly uses BarycentricMeshSystem) ----
def _lookup(theta, thetadot):
    """Evaluate the FittedValueIteration policy at (theta, thetadot).
    Uses the BarycentricMeshSystem directly, same approach as the simulation
    cell that achieves 0.0005 rad accuracy.
    """
    th = np.mod(theta, 2 * np.pi)
    td = np.clip(thetadot, -thetadot_max, thetadot_max)
    ctx = policy.CreateDefaultContext()
    policy.get_input_port(0).FixValue(ctx, BasicVector([th, td]))
    return float(policy.get_output_port(0).Eval(ctx)[0])


# ---- Build scene with standalone meshcat ----
vis = Visualizer()
vis.open()

vis["/Background"].set_property("top_color", [0.12, 0.12, 0.22])
vis["/Background"].set_property("bottom_color", [0.35, 0.40, 0.50])

# Axes: thin colored boxes
def _add_axis(name, size, offset, color):
    vis[name].set_object(meshcat_geom.Box(size),
                         meshcat_geom.MeshLambertMaterial(color=color))
    vis[name].set_transform(tf_meshcat.translation_matrix(offset))

_add_axis("/Axes/x", [2.0, 0.015, 0.015], [1.0, 0, 0], 0xff6666)
_add_axis("/Axes/y", [0.015, 2.0, 0.015], [0, -1.0, 0], 0x66ff66)
_add_axis("/Axes/z", [0.015, 0.015, 2.0], [0, 0, 1.0], 0x6666ff)

# Pivot
vis["/pivot"].set_object(
    meshcat_geom.Sphere(0.06), meshcat_geom.MeshPhongMaterial(color=0xdddddd))

# Rod — use a thin Box instead of Cylinder (avoids axis confusion)
vis["/rod"].set_object(
    meshcat_geom.Box([0.03, 1.0, 0.03]),
    meshcat_geom.MeshPhongMaterial(color=0x888888))

# Bob
vis["/bob"].set_object(
    meshcat_geom.Sphere(0.10), meshcat_geom.MeshPhongMaterial(color=0xff5533))

# Target at upright
vis["/target"].set_object(
    meshcat_geom.Sphere(0.07), meshcat_geom.MeshPhongMaterial(color=0x44ff44, opacity=0.5))
vis["/target"].set_transform(tf_meshcat.translation_matrix([0, 1.0, 0]))

# Camera
vis["/Cameras/default/rotated"].set_transform(
    tf_meshcat.translation_matrix([0, 0, 4.5]) @
    tf_meshcat.euler_matrix(-0.1, 0, 0))

print(f"MeshCat URL: {vis.url()}")

# ---- Pose update ----
def _update_pose(theta):
    """Rod from origin to bob. Visual length = 1.0."""
    bx = 1.0 * np.sin(theta)
    by = -1.0 * np.cos(theta)
    # Box center at rod midpoint, rotated by theta around Z
    vis["/rod"].set_transform(
        tf_meshcat.translation_matrix([bx / 2, by / 2, 0]) @
        tf_meshcat.rotation_matrix(theta, [0, 0, 1]))
    vis["/bob"].set_transform(tf_meshcat.translation_matrix([bx, by, 0]))

# Show initial pose
_update_pose(np.radians(18))

# ---- Simulation runner (step-by-step pydrake) ----
def run_interactive(theta0_rad, thetadot0_rad, duration=8.0):
    _update_pose(theta0_rad)
    _time.sleep(0.5)

    x = np.array([theta0_rad, thetadot0_rad])
    u = _lookup(theta0_rad, thetadot0_rad)

    dt_step, dt_render = 0.01, 0.08
    n_steps = int(duration / dt_step)
    next_render = 0.0
    t_wall = _time.time()

    for k in range(1, n_steps + 1):
        plant = PendulumPlant()
        context = plant.CreateDefaultContext()
        context.SetContinuousState(x)
        plant.get_input_port(0).FixValue(context, BasicVector([u]))
        sim = Simulator(plant, context)
        try:
            sim.get_mutable_integrator().set_fixed_step_mode(True)
        except AttributeError:
            pass
        sim.AdvanceTo(dt_step)
        x = sim.get_context().get_continuous_state_vector().CopyToVector()

        u = _lookup(x[0], x[1])

        t = k * dt_step
        if t >= next_render:
            _update_pose(np.mod(x[0], 2 * np.pi))
            next_render += dt_render
            e = _time.time() - t_wall
            if e < dt_render:
                _time.sleep(dt_render - e)
            t_wall = _time.time()

    err = abs(np.mod(x[0] - np.pi + np.pi, 2 * np.pi) - np.pi)
    return x[0], err

# ---- ipywidgets ----
theta_slider = widgets.FloatSlider(
    value=18, min=0, max=360, step=1,
    description='theta0 [deg]:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px'))
thetadot_slider = widgets.FloatSlider(
    value=0, min=-180, max=180, step=5,
    description='thetadot0 [deg/s]:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px'))
dur_slider = widgets.FloatSlider(
    value=8, min=2, max=15, step=1,
    description='Duration [s]:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px'))
start_btn = widgets.Button(
    description='Start Simulation', button_style='success',
    layout=widgets.Layout(width='180px', height='45px'))
status_label = widgets.Label(value='Ready.')
progress_out = widgets.Output()

# Presets
preset_btns_list = []
for name, angle in [
    ('Near hanging (18 deg)', 18), ('45 deg', 45),
    ('Horizontal (90 deg)', 90), ('135 deg', 135),
    ('Near upright (162 deg)', 162),
]:
    btn = widgets.Button(description=name, layout=widgets.Layout(width='170px'))
    btn.on_click(lambda b, a=angle: setattr(theta_slider, 'value', a))
    preset_btns_list.append(btn)
preset_btns = widgets.HBox(preset_btns_list)

# Slider preview
def _on_slider(change):
    _update_pose(np.radians(theta_slider.value))
theta_slider.observe(_on_slider, names='value')

# Start callback
def on_start(b):
    th0 = np.radians(theta_slider.value)
    td0 = np.radians(thetadot_slider.value)
    dur = dur_slider.value
    start_btn.disabled = True
    status_label.value = f'Running: theta0={theta_slider.value} deg...'
    with progress_out:
        clear_output(wait=True)
        print(f'Starting from {theta_slider.value} deg, duration={dur}s ...')
    try:
        th_final, err = run_interactive(th0, td0, dur)
        with progress_out:
            deg_err = np.degrees(err)
            print(f'Final angle error: {deg_err:.1f} deg')
            if err < 0.15:
                print('成功稳定在目标点！')
            else:
                print(f'Off by {deg_err:.0f} deg')
    except Exception as e:
        with progress_out:
            import traceback
            print(f'ERROR: {e}')
            traceback.print_exc()
    status_label.value = 'Done.'
    start_btn.disabled = False

start_btn.on_click(on_start)

# Display UI
display(widgets.HTML(
    f'<h3>交互式单摆控制</h3>'
    f'<p>设置初始状态, 点击 <b>Start</b>. '
    f'<a href="{vis.url()}" target="_blank">打开 MeshCat</a> 观看</p>'))
display(widgets.HTML('<b>预设:</b>'))
display(preset_btns)
display(theta_slider)
display(thetadot_slider)
display(dur_slider)
display(widgets.HBox([start_btn, status_label]))
display(progress_out)